In [2]:
import pandas as pd
import numpy as np 

In [14]:
df = pd.read_csv("cleaned_dataset.csv")
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount_usd,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [15]:
df["promo_code_used"].unique()

array(['Yes', 'No'], dtype=object)

## Dependency Score

This feature estimates how dependent a customer is on promotions and discounts.

### Category Logic
- 0 → No promo code and no discount used
- 1 → Either promo code OR discount used
- 2 → Both promo code and discount used

### Business Purpose
To identify whether customer purchases are driven by genuine brand loyalty or promotional incentives.

In [16]:
df['dependency_score'] = (
    df['promo_code_used'].map({'Yes':1, 'No':0}) +
    df['discount_applied'].map({'Yes':1, 'No':0})
)

## Value Tier

This feature classifies customers based on purchase amount.

### Category Logic
- Purchase Amount < 50 → Low Value
- Purchase Amount between 50 and 90 → Medium Value
- Purchase Amount ≥ 90 → High Value

### Business Purpose
To identify high-value customers and distinguish premium buyers from low-spending customers.

In [32]:
df['purchase_amount_usd'].describe()

count    3900.000000
mean       59.764359
std        23.685392
min        20.000000
25%        39.000000
50%        60.000000
75%        81.000000
max       100.000000
Name: purchase_amount_usd, dtype: float64

In [27]:
def value_tier(x):
    if x <= 40:
        return 'Low Value'
    elif x <= 80:
        return 'Medium Value'
    else:
        return 'High Value'

df['value_tier'] = df['purchase_amount_usd'].apply(value_tier)

In [31]:
df['value_tier'].value_counts()

value_tier
Medium Value    1862
Low Value       1060
High Value       978
Name: count, dtype: int64

## Satisfaction Flag

This feature indicates whether the customer appears satisfied based on review ratings.

### Category Logic
- Review Rating ≥ 4 → Satisfied
- Review Rating < 4 → Unsatisfied

### Business Purpose
To identify dissatisfied customers and potential retention issues.

In [33]:
df['satisfaction_flag'] = df['review_rating'].apply(
    lambda x: 'Satisfied' if x >= 4 else 'Unsatisfied'
)

In [34]:
df['satisfaction_flag'].value_counts()

satisfaction_flag
Unsatisfied    2266
Satisfied      1634
Name: count, dtype: int64

## Loyalty Segment

This feature categorizes customers according to their purchase history.

### Category Logic
- Previous Purchases < 13 → New Customer
- Previous Purchases between 13 and 38 → Regular Customer
- Previous Purchases ≥ 38 → Loyal Customer

### Business Purpose
To separate loyal repeat customers from occasional or new customers.

In [36]:
df['previous_purchases'].describe()

count    3900.000000
mean       25.351538
std        14.447125
min         1.000000
25%        13.000000
50%        25.000000
75%        38.000000
max        50.000000
Name: previous_purchases, dtype: float64

In [37]:
def loyalty_segment(x):
    if x <= 13:
        return 'New Customer'
    elif x <= 38:
        return 'Regular Customer'
    else:
        return 'Loyal Customer'

df['loyalty_segment'] = df['previous_purchases'].apply(loyalty_segment)

## Retention Risk

This feature identifies customers who may be at risk of discontinuing purchases.

### Category Logic
Customer is marked as High Risk when:
- Review Rating < 3
- Dependency Score ≥ 1
- Previous Purchases < 10

Otherwise:
- Low Risk

### Business Purpose
To help the business proactively target customers likely to churn.

In [43]:
df['retention_risk'] = np.where(
    (df['review_rating'] < 3) &
    (df['dependency_score'] >= 1) &
    (df['previous_purchases'] < 10),
    'High Risk',
    'Low Risk'
)

## Age Group

This feature groups customers into age segments.

### Category Logic
- Age ≤ 25 → Young Adult
- Age between 26 and 40 → Adult
- Age > 40 → Mature

### Business Purpose
To identify which age groups contribute most to customer value and retention.

In [44]:
def age_group(age):
    if age <= 25:
        return 'Young Adult'
    elif age <= 40:
        return 'Adult'
    else:
        return 'Mature'

df['age_group'] = df['age'].apply(age_group)

## Promo Sensitivity

This feature classifies customers according to their promotional dependency level.

### Category Logic
- Dependency Score = 0 → Low
- Dependency Score = 1 → Medium
- Dependency Score = 2 → High

### Business Purpose
To identify which customer groups are highly discount-sensitive.

In [45]:
def promo_sensitivity(score):
    if score == 0:
        return 'Low'
    elif score == 1:
        return 'Medium'
    else:
        return 'High'

df['promo_sensitivity'] = df['dependency_score'].apply(promo_sensitivity)

## Premium Customer

This feature flags high-value loyal customers.

### Category Logic
Customer is marked as Premium Customer when:
- Value Tier = High Value
- Loyalty Segment = Loyal Customer

Otherwise:
- No

### Business Purpose
To identify customers who contribute significantly to long-term revenue.

In [46]:
df['premium_customer'] = np.where(
    (df['value_tier'] == 'High Value') &
    (df['loyalty_segment'] == 'Loyal Customer'),
    'Yes',
    'No'
)

In [47]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount_usd,location,size,color,season,...,payment_method,frequency_of_purchases,dependency_score,value_tier,satisfaction_flag,loyalty_segment,retention_risk,age_group,promo_sensitivity,premium_customer
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Venmo,Fortnightly,2,Medium Value,Unsatisfied,Regular Customer,Low Risk,Mature,High,No
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Cash,Fortnightly,2,Medium Value,Unsatisfied,New Customer,Low Risk,Young Adult,High,No
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Credit Card,Weekly,2,Medium Value,Unsatisfied,Regular Customer,Low Risk,Mature,High,No
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,PayPal,Weekly,2,High Value,Unsatisfied,Loyal Customer,Low Risk,Young Adult,High,Yes
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,PayPal,Annually,2,Medium Value,Unsatisfied,Regular Customer,Low Risk,Mature,High,No


In [48]:
df.to_csv("engineered_dataset.csv", index=False)